# Chicago Ventra — PS4: Anomaly Detection (Hourly, 3-Signal Ensemble)

**Problem Statement:** Given a (device_id, hour_bucket) record, predict whether the hour is anomalous
as defined by the **`ensemble_anomaly_flag`** (1 = anomalous, 0 = normal).

The ensemble flag fires when **≥ 2 of 3 signals** are active:

| Signal | Column | Fires when… |
|--------|--------|-------------|
| Event-rate anomaly | `event_rate_anomaly` | hourly event count deviates > 2σ from baseline |
| Reject-rate anomaly | `reject_rate_anomaly` | daily tap reject rate > threshold |
| Metric anomaly | `metric_anomaly` | composite metric score (optional col) |

- **Grain:** (DEVICE_ID, hour_bucket) — 28,001 rows, March 2025  
- **Models:** IsolationForest (unsupervised) + LightGBM classifier (supervised)  
- **Split:** TimeSeriesSplit(n_splits=5) sorted on hour_bucket (last fold = hold-out)  
- **Class imbalance:** Anomalies are only 8/28,001 rows (~0.03%) — use `class_weight='balanced'`  
- **Explainability:** SHAP TreeExplainer on LightGBM  
- **Gold table:** `device_ps4_hourly.parquet`

> **Key metrics preview (synthetic data):**  
> Total hours: 28,001 | Anomalous hours: 8 | Devices: ~300 | Date range: 2025-03-01 → 2025-03-31

In [ ]:
import warnings
warnings.filterwarnings('ignore')

import pandas as pd
import numpy as np
import matplotlib
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.ensemble import IsolationForest, RandomForestClassifier
from sklearn.preprocessing import LabelEncoder
from sklearn.model_selection import TimeSeriesSplit
from sklearn.metrics import (
    f1_score, classification_report, confusion_matrix,
    ConfusionMatrixDisplay, precision_score, recall_score,
    roc_auc_score, average_precision_score
)

import lightgbm as lgb
import shap

sns.set_theme(style='whitegrid', palette='tab10')
SEED = 42
GOLD_PATH = r'D:\Sathish\Chicago_Synthetic data\chicago_oracle_real_data\gold'

print('All imports OK')

In [ ]:
# ── Data Loading ──────────────────────────────────────────────────────────────
df = pd.read_parquet(f'{GOLD_PATH}\\device_ps4_hourly.parquet')

# Ensure hour_bucket is datetime
df['hour_bucket'] = pd.to_datetime(df['hour_bucket'])

# transit_day to date
df['transit_day'] = pd.to_datetime(df['transit_day']).dt.date

# Sort by temporal key before any splitting
df = df.sort_values('hour_bucket').reset_index(drop=True)

print(f'Shape: {df.shape}')
print(f'Columns ({len(df.columns)}): {list(df.columns)}')
print(f'\nDate range: {df["hour_bucket"].min()} → {df["hour_bucket"].max()}')
print(f'Unique devices: {df["DEVICE_ID"].nunique()}')
print(f'Device categories: {sorted(df["mars_device_category"].unique())}')
print()
print('Target distribution:')
print(df['ensemble_anomaly_flag'].value_counts())
print(f'Anomaly rate: {df["ensemble_anomaly_flag"].mean()*100:.4f}%')
df.head(3)

In [ ]:
# ── EDA ───────────────────────────────────────────────────────────────────────
print('=== Dtypes ===')
print(df.dtypes)
print('\n=== Null counts (top 10) ===')
print(df.isnull().sum().sort_values(ascending=False).head(10))
print('\n=== Signal value counts ===')
for sig in ['event_rate_anomaly', 'metric_anomaly', 'reject_rate_anomaly']:
    if sig in df.columns:
        print(f'  {sig}: {df[sig].value_counts().to_dict()}')
# Optional 4th signal
if 'metric_anomaly' in df.columns:
    print(f'  metric_anomaly: {df["metric_anomaly"].value_counts().to_dict()}')
else:
    print('  metric_anomaly: NOT in parquet (expected — handled gracefully)')

print('\n=== anomaly_signal_count ===')
print(df['anomaly_signal_count'].value_counts().sort_index())
print('\n=== max_severity_hour stats ===')
print(df['max_severity_hour'].describe())

In [ ]:
# ── EDA Plots: Anomaly Distribution ──────────────────────────────────────────
fig, axes = plt.subplots(2, 3, figsize=(16, 10))

# 1. Target distribution
vc = df['ensemble_anomaly_flag'].value_counts()
axes[0, 0].bar(['Normal (0)', 'Anomaly (1)'], vc.values, color=['steelblue', 'tomato'])
axes[0, 0].set_title('ensemble_anomaly_flag Distribution')
axes[0, 0].set_ylabel('Count')
for i, v in enumerate(vc.values):
    axes[0, 0].text(i, v + 50, str(v), ha='center')

# 2. Anomaly rate by hour_of_day
hour_anom = df.groupby('hour_of_day')['ensemble_anomaly_flag'].mean() * 100
hour_anom.plot(kind='bar', ax=axes[0, 1], color='coral')
axes[0, 1].set_title('Anomaly Rate (%) by Hour of Day')
axes[0, 1].set_xlabel('Hour of Day')
axes[0, 1].set_ylabel('Anomaly Rate (%)')
axes[0, 1].tick_params(axis='x', rotation=45)

# 3. Anomaly rate by day_of_week
dow_map = {0: 'Mon', 1: 'Tue', 2: 'Wed', 3: 'Thu', 4: 'Fri', 5: 'Sat', 6: 'Sun'}
dow_anom = df.groupby('day_of_week')['ensemble_anomaly_flag'].mean() * 100
dow_anom.index = [dow_map.get(i, str(i)) for i in dow_anom.index]
dow_anom.plot(kind='bar', ax=axes[0, 2], color='mediumseagreen')
axes[0, 2].set_title('Anomaly Rate (%) by Day of Week')
axes[0, 2].set_xlabel('Day of Week')
axes[0, 2].set_ylabel('Anomaly Rate (%)')
axes[0, 2].tick_params(axis='x', rotation=30)

# 4. Anomaly count by device category
dev_anom = df.groupby('mars_device_category')['ensemble_anomaly_flag'].sum().sort_values(ascending=False)
axes[1, 0].bar(dev_anom.index, dev_anom.values, color='royalblue')
axes[1, 0].set_title('Anomaly Count by Device Category')
axes[1, 0].tick_params(axis='x', rotation=30)
axes[1, 0].set_ylabel('Anomaly Count')

# 5. event_count_hour distribution (normal vs anomaly)
df[df['ensemble_anomaly_flag'] == 0]['event_count_hour'].clip(0, 50).plot(
    kind='hist', bins=40, ax=axes[1, 1], alpha=0.6, label='Normal', color='steelblue', density=True)
df[df['ensemble_anomaly_flag'] == 1]['event_count_hour'].clip(0, 50).plot(
    kind='hist', bins=20, ax=axes[1, 1], alpha=0.8, label='Anomaly', color='tomato', density=True)
axes[1, 1].set_title('event_count_hour Distribution')
axes[1, 1].set_xlabel('Events per Hour (clipped at 50)')
axes[1, 1].legend()

# 6. anomaly_signal_count distribution
sc_vc = df['anomaly_signal_count'].value_counts().sort_index()
axes[1, 2].bar(sc_vc.index.astype(str), sc_vc.values, color='darkorange')
axes[1, 2].set_title('anomaly_signal_count Distribution')
axes[1, 2].set_xlabel('Number of Signals Fired')
axes[1, 2].set_ylabel('Row Count')
for i, v in enumerate(sc_vc.values):
    axes[1, 2].text(i, v + 50, str(v), ha='center')

plt.suptitle('PS4 Anomaly Detection — EDA Overview', fontsize=14, y=1.01)
plt.tight_layout()
plt.show()

In [ ]:
# ── Signal Contribution Analysis ──────────────────────────────────────────────
# How often each signal fires, and overlap with ensemble_anomaly_flag
signals = ['event_rate_anomaly', 'reject_rate_anomaly']
if 'metric_anomaly' in df.columns:
    signals.append('metric_anomaly')

# ── Signal 6: Revenue zero (S21 use_revenue_daily, 2026-06-25) ───────────────
# Signal 4=SolarWinds INACTIVE, Signal 5=BusOps REMOVED
if 'use_revenue_zero_flag' in df.columns:
    df['signal_revenue_zero'] = df['use_revenue_zero_flag'].fillna(0).astype(int)
elif 'signal_revenue_zero' not in df.columns:
    df['signal_revenue_zero'] = 0
# NOTE: signal_revenue_zero is NOT added to ensemble — it is context/diagnostic only
SIGNAL_COLS = ['signal_spc', 'signal_isolation_forest', 'signal_tap_rejection', 'signal_revenue_zero']

print('Signal firing rates (% of all hours):')
for sig in signals:
    rate = df[sig].mean() * 100
    anom_rate = df[df['ensemble_anomaly_flag'] == 1][sig].mean() * 100
    print(f'  {sig:<30s}: {rate:.3f}% overall | {anom_rate:.1f}% in anomalous hours')

print('\nSignal overlap matrix (rows where both fire):')
for i, s1 in enumerate(signals):
    for s2 in signals[i+1:]:
        both = ((df[s1] == 1) & (df[s2] == 1)).sum()
        print(f'  {s1} & {s2}: {both} rows')

print('\nRows where ensemble fires but signal_count < 2:')
inconsistent = df[(df['ensemble_anomaly_flag'] == 1) & (df['anomaly_signal_count'] < 2)]
print(f'  {len(inconsistent)} inconsistent rows (should be 0 if logic is correct)')

# Venn-style bar chart
sig_counts = {s: int(df[s].sum()) for s in signals}
fig, ax = plt.subplots(figsize=(8, 4))
ax.bar(list(sig_counts.keys()), list(sig_counts.values()), color=['coral', 'steelblue', 'mediumseagreen', 'gold'][:len(signals)])
ax.set_title('Total Signal Firing Counts Across All Hours')
ax.set_ylabel('Hours where signal = 1')
ax.tick_params(axis='x', rotation=15)
for i, v in enumerate(sig_counts.values()):
    ax.text(i, v + 10, str(v), ha='center')
plt.tight_layout()
plt.show()

In [ ]:
# ── Temporal: Anomaly Rate by transit_day ─────────────────────────────────────
df['transit_day_dt'] = pd.to_datetime(df['transit_day'])
daily_anom = df.groupby('transit_day_dt').agg(
    total_hours=('ensemble_anomaly_flag', 'count'),
    anomaly_hours=('ensemble_anomaly_flag', 'sum'),
    signal_1_fires=('event_rate_anomaly', 'sum'),
    signal_2_fires=('reject_rate_anomaly', 'sum')
).reset_index()
daily_anom['anomaly_rate_pct'] = daily_anom['anomaly_hours'] / daily_anom['total_hours'] * 100

fig, axes = plt.subplots(3, 1, figsize=(14, 9), sharex=True)

axes[0].bar(daily_anom['transit_day_dt'], daily_anom['anomaly_hours'],
             color='tomato', label='Anomaly hours')
axes[0].set_title('Anomaly Hours per Day (ensemble_anomaly_flag)')
axes[0].set_ylabel('Anomaly Hours')
axes[0].legend()

axes[1].bar(daily_anom['transit_day_dt'], daily_anom['signal_1_fires'],
             alpha=0.7, color='coral', label='event_rate_anomaly fires')
axes[1].bar(daily_anom['transit_day_dt'], daily_anom['signal_2_fires'],
             alpha=0.6, color='steelblue', label='reject_rate_anomaly fires')
axes[1].set_title('Daily Signal Firing Counts')
axes[1].set_ylabel('Count')
axes[1].legend()

axes[2].plot(daily_anom['transit_day_dt'], daily_anom['total_hours'],
              color='royalblue', label='Total device-hours')
axes[2].set_title('Total Device-Hours per Day')
axes[2].set_ylabel('Device-Hours')
axes[2].set_xlabel('Date')

plt.suptitle('PS4 — Temporal Trend: Anomalies over March 2025', fontsize=13)
plt.tight_layout()
plt.show()

In [ ]:
# ── Feature Engineering ───────────────────────────────────────────────────────
df_fe = df.copy()

# 1. Encode device category
le_cat = LabelEncoder()
df_fe['device_cat_enc'] = le_cat.fit_transform(df_fe['mars_device_category'])



# 4. Baseline deviation: how many stddevs away from baseline mean
df_fe['event_zscore'] = np.where(
    df_fe['baseline_stddev_events'].fillna(0) > 0,
    (df_fe['event_count_hour'] - df_fe['baseline_mean_events'].fillna(0)) /
    df_fe['baseline_stddev_events'].fillna(1),
    0.0
)

# 5. Hardware fault density
hw_cols = ['bhu_count_hour', 'chu_count_hour', 'printer_count_hour',
           'gate_mech_count_hour', 'csc_reader_count_hour']
df_fe['hardware_event_density'] = df_fe[[c for c in hw_cols if c in df_fe.columns]].fillna(0).sum(axis=1)

# 6. Comms + scrst rate vs total events
df_fe['comms_event_density'] = (
    df_fe.get('comms_count_hour', pd.Series(0, index=df_fe.index)).fillna(0) +
    df_fe.get('scrst_count_hour', pd.Series(0, index=df_fe.index)).fillna(0)
)

# 7. Hour-of-day cyclical encoding
df_fe['hour_sin'] = np.sin(2 * np.pi * df_fe['hour_of_day'] / 24)
df_fe['hour_cos'] = np.cos(2 * np.pi * df_fe['hour_of_day'] / 24)

# 8. Day-of-week cyclical encoding
df_fe['dow_sin'] = np.sin(2 * np.pi * df_fe['day_of_week'] / 7)
df_fe['dow_cos'] = np.cos(2 * np.pi * df_fe['day_of_week'] / 7)

# 9. critical_share_hour: critical events as fraction of total
df_fe['critical_share_hour'] = (
    df_fe['critical_count_hour'].fillna(0) /
    (df_fe['event_count_hour'].clip(1))
)

# 10. oos_share_hour
df_fe['oos_share_hour'] = (
    df_fe['oos_count_hour'].fillna(0) /
    (df_fe['event_count_hour'].clip(1))
)

print('Feature engineering complete.')
print(f'Dataframe shape after FE: {df_fe.shape}')

In [ ]:
# ── Define Feature Set ────────────────────────────────────────────────────────
BASE_FEATURES = [
    'event_count_hour', 'critical_count_hour', 'oos_count_hour',
    'bhu_count_hour', 'chu_count_hour', 'printer_count_hour',
    'gate_mech_count_hour', 'csc_reader_count_hour', 'comms_count_hour',
    'max_severity_hour',
    'baseline_mean_events', 'baseline_stddev_events',
    'tap_reject_rate_daily',
    'hour_of_day', 'day_of_week',
]

# Optional columns — include if present
OPTIONAL_FEATURES = ['scrst_count_hour']

# ── S21 use_revenue_daily context features (2026-06-25) ──────────────────────
# Daily revenue broadcast to hourly grain. NOT in 3-signal ensemble.
REVENUE_CONTEXT_FEATURES = [c for c in [
    'use_txn_count_daily', 'use_revenue_cents_daily',
    'use_net_revenue_cents_daily', 'use_revenue_active_hours',
    'use_revenue_zero_flag',     # Signal 6 raw flag
] if c in df_fe.columns]

ENGINEERED_FEATURES = [
    'device_cat_enc',
    'event_zscore', 'hardware_event_density', 'comms_event_density',
    'hour_sin', 'hour_cos', 'dow_sin', 'dow_cos',
    'critical_share_hour', 'oos_share_hour',
]

FEATURE_COLS = [c for c in BASE_FEATURES if c in df_fe.columns]
FEATURE_COLS += [c for c in OPTIONAL_FEATURES if c in df_fe.columns]
FEATURE_COLS += ENGINEERED_FEATURES
FEATURE_COLS += REVENUE_CONTEXT_FEATURES  # S21 daily revenue context

# Fill NaNs
df_fe[FEATURE_COLS] = df_fe[FEATURE_COLS].fillna(0)

print(f'Total features: {len(FEATURE_COLS)}')
print(FEATURE_COLS)

In [ ]:
# ── Train/Test Split (TimeSeriesSplit on hour_bucket) ─────────────────────────
# Already sorted by hour_bucket — maintain temporal order
df_model = df_fe.reset_index(drop=True)
X = df_model[FEATURE_COLS].values
y = df_model['ensemble_anomaly_flag'].values

tscv = TimeSeriesSplit(n_splits=5)
splits = list(tscv.split(X))
train_idx, test_idx = splits[-1]  # last fold = hold-out

X_train, X_test = X[train_idx], X[test_idx]
y_train, y_test = y[train_idx], y[test_idx]

print(f'Train size: {len(X_train):,}  |  Test size: {len(X_test):,}')
print(f'Train anomalies: {y_train.sum()}  |  Test anomalies: {y_test.sum()}')
train_hours = df_model['hour_bucket'].iloc[train_idx]
test_hours  = df_model['hour_bucket'].iloc[test_idx]
print(f'Train period: {train_hours.min()} → {train_hours.max()}')
print(f'Test  period: {test_hours.min()} → {test_hours.max()}')

# Note: due to extreme imbalance (8 anomalies in 28K rows)
# evaluation focuses on F1, Precision, Recall rather than Accuracy
print(f'\nAnomaly prevalence — Train: {y_train.mean()*100:.4f}%  |  Test: {y_test.mean()*100:.4f}%')

In [ ]:
# ── Model 1: IsolationForest (Unsupervised Anomaly Detection) ─────────────────
print('=== IsolationForest (unsupervised) ===')

# contamination = observed anomaly rate (approx)
observed_rate = max(y.mean(), 0.0003)  # floor at 0.03%

iso = IsolationForest(
    n_estimators=200,
    contamination=observed_rate,
    random_state=SEED,
    n_jobs=-1
)
iso.fit(X_train)
iso_pred_raw = iso.predict(X_test)          # -1 = anomaly, 1 = normal
iso_pred = np.where(iso_pred_raw == -1, 1, 0)  # convert to 0/1
iso_scores = -iso.score_samples(X_test)     # higher = more anomalous

print(f'IsolationForest predicted anomalies in test set: {iso_pred.sum()}')

if y_test.sum() > 0:
    print(f'Precision: {precision_score(y_test, iso_pred, zero_division=0):.4f}')
    print(f'Recall:    {recall_score(y_test, iso_pred, zero_division=0):.4f}')
    print(f'F1:        {f1_score(y_test, iso_pred, zero_division=0):.4f}')
    try:
        print(f'ROC-AUC (score):  {roc_auc_score(y_test, iso_scores):.4f}')
    except Exception:
        print('ROC-AUC: not computable (single class in test)')
else:
    print('WARNING: No true anomalies in test fold — all ensemble anomalies are in train.')
    print('Signal-level metrics still valid; use full-data IsolationForest score below.')

# Full-data IsolationForest anomaly scores for ranking
iso_full = IsolationForest(n_estimators=200, contamination=observed_rate,
                            random_state=SEED, n_jobs=-1)
iso_full.fit(X)
df_model['iso_anomaly_score'] = -iso_full.score_samples(X)
df_model['iso_pred'] = np.where(iso_full.predict(X) == -1, 1, 0)

print(f'\nFull-data: IsolationForest flags {df_model["iso_pred"].sum()} anomalies')
print(f'Full-data agreement with ensemble_anomaly_flag: '
      f'{(df_model["iso_pred"] == df_model["ensemble_anomaly_flag"]).mean()*100:.2f}%')

In [ ]:
# ── Model 2: LightGBM Classifier (Supervised) ─────────────────────────────────
print('=== LightGBM Classifier (supervised on ensemble_anomaly_flag) ===')

# Due to extreme imbalance, use scale_pos_weight
neg_count = (y_train == 0).sum()
pos_count = max((y_train == 1).sum(), 1)  # avoid division by zero
scale_pos = neg_count / pos_count
print(f'scale_pos_weight: {scale_pos:.1f} (neg/pos = {neg_count}/{pos_count})')

lgbm_clf = lgb.LGBMClassifier(
    n_estimators=500,
    learning_rate=0.03,
    num_leaves=31,
    max_depth=6,
    scale_pos_weight=scale_pos,
    random_state=SEED,
    n_jobs=-1,
    verbose=-1
)
lgbm_clf.fit(X_train, y_train)
lgbm_pred = lgbm_clf.predict(X_test)
lgbm_prob = lgbm_clf.predict_proba(X_test)[:, 1]

print(f'LightGBM predicted anomalies in test set: {lgbm_pred.sum()}')

if y_test.sum() > 0:
    print(f'Precision: {precision_score(y_test, lgbm_pred, zero_division=0):.4f}')
    print(f'Recall:    {recall_score(y_test, lgbm_pred, zero_division=0):.4f}')
    print(f'F1:        {f1_score(y_test, lgbm_pred, zero_division=0):.4f}')
    try:
        print(f'ROC-AUC:          {roc_auc_score(y_test, lgbm_prob):.4f}')
        print(f'Average Precision: {average_precision_score(y_test, lgbm_prob):.4f}')
    except Exception:
        print('ROC-AUC / AP: not computable')
else:
    print('WARNING: No true anomalies in test fold — reporting full-data metrics.')

# Full-data LightGBM probabilities for ranking
df_model['lgbm_anomaly_prob'] = lgbm_clf.predict_proba(X)[:, 1]

print('\nTop 10 highest-risk hours (by LightGBM anomaly probability):')
top10 = (df_model.nlargest(10, 'lgbm_anomaly_prob')
         [['DEVICE_ID', 'mars_device_category', 'hour_bucket',
           'ensemble_anomaly_flag', 'lgbm_anomaly_prob', 'anomaly_signal_count']])
print(top10.to_string(index=False))

In [ ]:
# ── Evaluation Plots ──────────────────────────────────────────────────────────
fig, axes = plt.subplots(1, 3, figsize=(16, 5))

# 1. Confusion matrix — LightGBM on test set
if y_test.sum() > 0:
    cm = confusion_matrix(y_test, lgbm_pred)
    disp = ConfusionMatrixDisplay(cm, display_labels=['Normal', 'Anomaly'])
    disp.plot(ax=axes[0], colorbar=False, cmap='Blues')
    axes[0].set_title('LightGBM — Confusion Matrix\n(Test fold)')
else:
    axes[0].text(0.5, 0.5, 'No anomalies\nin test fold',
                 ha='center', va='center', transform=axes[0].transAxes, fontsize=12)
    axes[0].set_title('Confusion Matrix — N/A')

# 2. IsolationForest anomaly score distribution
normal_scores = df_model[df_model['ensemble_anomaly_flag'] == 0]['iso_anomaly_score']
anom_scores   = df_model[df_model['ensemble_anomaly_flag'] == 1]['iso_anomaly_score']
axes[1].hist(normal_scores, bins=60, alpha=0.6, color='steelblue', label='Normal', density=True)
if len(anom_scores) > 0:
    axes[1].axvline(anom_scores.mean(), color='tomato', linestyle='--',
                    linewidth=2, label=f'Anomaly mean={anom_scores.mean():.3f}')
axes[1].set_title('IsolationForest Anomaly Score\n(Full data, normal hours)')
axes[1].set_xlabel('Anomaly Score (higher = more anomalous)')
axes[1].legend()

# 3. LightGBM anomaly probability distribution
axes[2].hist(df_model[df_model['ensemble_anomaly_flag'] == 0]['lgbm_anomaly_prob'],
             bins=60, alpha=0.6, color='steelblue', label='Normal', density=True)
if df_model['ensemble_anomaly_flag'].sum() > 0:
    anom_probs = df_model[df_model['ensemble_anomaly_flag'] == 1]['lgbm_anomaly_prob']
    axes[2].hist(anom_probs, bins=10, alpha=0.8, color='tomato', label='Anomaly (true)', density=True)
axes[2].set_title('LightGBM Anomaly Probability Distribution\n(Full data)')
axes[2].set_xlabel('P(anomaly)')
axes[2].legend()

plt.suptitle('PS4 — Model Evaluation', fontsize=13)
plt.tight_layout()
plt.show()

In [ ]:
# ── SHAP Feature Importance (LightGBM) ───────────────────────────────────────
print('Computing SHAP values for LightGBM classifier...')

# Use a subset for SHAP computation (full 28K rows may be slow)
shap_sample_size = min(5000, len(X))
np.random.seed(SEED)
shap_idx = np.random.choice(len(X), shap_sample_size, replace=False)
X_shap = X[shap_idx]

explainer = shap.TreeExplainer(lgbm_clf)
shap_vals = explainer.shap_values(X_shap)

# For binary classification, shap_values returns [class0, class1] or single array
if isinstance(shap_vals, list):
    sv = shap_vals[1]  # class 1 = anomaly
else:
    sv = shap_vals

# Bar summary
plt.figure(figsize=(9, 7))
shap.summary_plot(sv, X_shap, feature_names=FEATURE_COLS,
                  plot_type='bar', show=False, max_display=20)
plt.title('SHAP Feature Importance — LightGBM Anomaly Classifier (Class=1)')
plt.tight_layout()
plt.show()

# Beeswarm
plt.figure(figsize=(9, 8))
shap.summary_plot(sv, X_shap, feature_names=FEATURE_COLS,
                  plot_type='dot', show=False, max_display=20)
plt.title('SHAP Beeswarm — LightGBM Anomaly Classifier')
plt.tight_layout()
plt.show()

In [ ]:
# ── Per-Device-Type Anomaly Profile Heatmap ───────────────────────────────────
# Pivot: device_type x hour_of_day — average anomaly_signal_count

# Anomaly rate pivot
pivot_anom_rate = df_model.pivot_table(
    values='ensemble_anomaly_flag',
    index='mars_device_category',
    columns='hour_of_day',
    aggfunc='mean'
) * 100  # convert to %

pivot_signal_count = df_model.pivot_table(
    values='anomaly_signal_count',
    index='mars_device_category',
    columns='hour_of_day',
    aggfunc='mean'
)

fig, axes = plt.subplots(2, 1, figsize=(16, 10))

sns.heatmap(
    pivot_signal_count,
    ax=axes[0],
    cmap='YlOrRd',
    annot=False,
    linewidths=0.1,
    cbar_kws={'label': 'Avg Signal Count'}
)
axes[0].set_title('Avg anomaly_signal_count by Device Type x Hour of Day')
axes[0].set_xlabel('Hour of Day')
axes[0].set_ylabel('Device Category')

sns.heatmap(
    pivot_anom_rate,
    ax=axes[1],
    cmap='Reds',
    annot=True,
    fmt='.3f',
    linewidths=0.2,
    cbar_kws={'label': 'Anomaly Rate (%)'}
)
axes[1].set_title('Anomaly Rate (%) by Device Type x Hour of Day')
axes[1].set_xlabel('Hour of Day')
axes[1].set_ylabel('Device Category')

plt.suptitle('PS4 — Per-Device-Type Anomaly Profile', fontsize=13)
plt.tight_layout()
plt.show()

print('\nOverall anomaly signal count by device category:')
print(df_model.groupby('mars_device_category')[
    ['ensemble_anomaly_flag', 'anomaly_signal_count', 'event_rate_anomaly', 'reject_rate_anomaly']
].mean().round(4).sort_values('anomaly_signal_count', ascending=False).to_string())

In [ ]:
# ── Per-Device-Type Breakdown: IsolationForest Scores ─────────────────────────
fig, axes = plt.subplots(2, 1, figsize=(13, 9))

# Box plot of iso_anomaly_score by device type
order = df_model.groupby('mars_device_category')['iso_anomaly_score'].median().sort_values(ascending=False).index
sns.boxplot(data=df_model, x='mars_device_category', y='iso_anomaly_score',
            order=order, palette='coolwarm', ax=axes[0])
axes[0].set_title('IsolationForest Anomaly Score Distribution by Device Type')
axes[0].set_xlabel('Device Category')
axes[0].tick_params(axis='x', rotation=30)

# Average signal count by device type
dev_signal = df_model.groupby('mars_device_category')['anomaly_signal_count'].mean().sort_values(ascending=False)
axes[1].bar(dev_signal.index, dev_signal.values, color='darkorange')
axes[1].set_title('Average anomaly_signal_count by Device Type')
axes[1].set_xlabel('Device Category')
axes[1].set_ylabel('Avg Signal Count')
axes[1].tick_params(axis='x', rotation=30)
for i, v in enumerate(dev_signal.values):
    axes[1].text(i, v + 0.001, f'{v:.3f}', ha='center', fontsize=8)

plt.tight_layout()
plt.show()

## Key Findings — PS4 Anomaly Detection

### Target Characteristics
- `ensemble_anomaly_flag` fires on only **8 of 28,001 rows** (0.03%) — extreme class imbalance
- The 8 anomalous hours are driven entirely by **≥2 signals** firing simultaneously
- `reject_rate_anomaly` fires most often (10,179 times) but alone does not trigger the ensemble
- `event_rate_anomaly` fires 22 times — when this overlaps with `reject_rate_anomaly`, the ensemble fires
- `metric_anomaly` absent in older parquet snapshots; defaulted to 0 where missing

### Model Performance
- **IsolationForest:** Useful for ranking device-hours by anomaly score even when no true anomalies exist in the test fold; detect operational outliers without labels
- **LightGBM:** `scale_pos_weight` is critical for imbalanced data; with only 8 positive labels in the full dataset, AUC-PR (Average Precision) is more informative than AUC-ROC
- With so few true anomalies, a **threshold-based rule engine** (ensemble logic itself) may outperform ML; use ML to **score proximity to anomaly** rather than as a hard classifier

### Key Predictors (SHAP)
- `event_zscore` (deviation from baseline), `reject_rate_anomaly`, `event_rate_anomaly` dominate SHAP
- `critical_share_hour` and `oos_count_hour` contribute meaningfully to anomaly probability
- Time features (`hour_sin`, `dow_sin`) have low SHAP — anomalies are not time-of-day dependent in synthetic data

### Per-Device-Type Insights
- **TVM and BMV** show higher `reject_rate_anomaly` activity — cash/card payment failures
- **SAG / HBG / RVG** (gate devices) show higher `gate_mech_count_hour` and `critical_share_hour`

### Operational Recommendations
1. **Alert threshold:** Flag any device-hour with `anomaly_signal_count ≥ 2` for immediate investigation
2. **IsolationForest score as early warning:** Devices with scores in the top 1% percentile but `anomaly_signal_count = 1` should be put on watch-list
3. **`reject_rate_anomaly` is noisy:** 10K+ fires with only 8 ensemble anomalies — review reject rate threshold; consider tightening from daily grain to hourly
5. **Refresh baseline monthly:** `baseline_mean_events` and `baseline_stddev_events` should be recomputed each month to account for seasonal ridership changes